In [1]:
import os
import matplotlib.pyplot as plt

import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
import cv2
from PIL import Image
import wandb

In [2]:
class Alexnet(nn.Module):
    def __init__(self,num_classes):
        super(Alexnet,self).__init__()
        self.layer1 = nn.Sequential(nn.Conv2d(in_channels=3,out_channels=96,kernel_size=11,stride=4,padding=0),
                                    nn.BatchNorm2d(num_features=96),
                                    nn.ReLU(),
                                    nn.MaxPool2d(kernel_size=3,stride=2))
        self.layer2 = nn.Sequential(nn.Conv2d(in_channels=96,out_channels=256,kernel_size=5,stride=1,padding=2),
                                    nn.BatchNorm2d(256),
                                    nn.ReLU(),
                                    nn.MaxPool2d(kernel_size=3,stride=2))
        self.layer3 = nn.Sequential(nn.Conv2d(in_channels=256,out_channels=384,kernel_size=3,padding=1,stride=1),
                                    nn.BatchNorm2d(384),
                                    nn.ReLU())
        self.layer4 = nn.Sequential(nn.Conv2d(in_channels=384,out_channels=384,kernel_size=3,stride=1,padding=1),
                                    nn.BatchNorm2d(384),
                                    nn.ReLU())
        self.layer5 = nn.Sequential(nn.Conv2d(in_channels=384,out_channels=256,kernel_size=3,stride=1,padding=1),
                                    nn.BatchNorm2d(256),
                                    nn.ReLU(),
                                    nn.MaxPool2d(kernel_size=3,stride=2),
                                    nn.Dropout(0.5))
        self.fc1    = nn.Sequential(nn.Dropout(0.5),
                                    nn.Linear(in_features=6400,out_features=4096),
                                    nn.ReLU())
        self.fc2    = nn.Sequential(nn.Dropout(0.5),
                                    nn.Linear(4096,4096),
                                    nn.ReLU())
        self.fc3    = nn.Sequential(nn.Linear(4096,num_classes))

    def forward(self,x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = out.view(out.size(0),-1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)

        return out




def load_labels(labels_csv_path):
    labels = {}
    if not os.path.isfile(labels_csv_path):
        return labels

    with open(labels_csv_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = [part.strip() for part in line.split(",", 1)]
            if len(parts) < 2:
                continue
            try:
                classid = int(parts[0])
            except ValueError:
                continue
            labels[classid] = parts[1]
    return labels



def count_images_by_class(folder_path):
    """
    Count images per class in a folder where filenames are like:
    00x_imagename.jpg, with x = class id (0..7).
    Display a bar chart with matplotlib and annotate each bar with the class label.
    """
    counts = {i: {"count": 0, "paths": []} for i in range(8)}

    for filename in os.listdir(folder_path):
        filepath = os.path.join(folder_path, filename)
        if not os.path.isfile(filepath):
            continue

        if len(filename) >= 4 and filename[3] == "_":
            try:
                classid = int(filename[2])
            except ValueError:
                continue
            if 0 <= classid <= 7:
                counts[classid]["count"] += 1
                counts[classid]["paths"].append(filepath)

    labels_csv = os.path.join(os.path.dirname(os.path.dirname(folder_path)), "labels.csv")
    labels = load_labels(labels_csv)

    class_ids = sorted(counts.keys())
    image_counts = [counts[classid]["count"] for classid in class_ids]
    label_texts = [labels.get(classid, f"class {classid}") for classid in class_ids]

    image_paths = []
    image_labels = []
    for classid in class_ids:
        label = labels.get(classid, f"class {classid}")
        print(f"class {classid} ({label}): {counts[classid]['count']} images")
        for path in counts[classid]["paths"]:
            image_paths.append(path)
            image_labels.append(classid)
    print("*****************************")

    return counts, image_paths, image_labels



config = {"epochs":250,
          "lr":0.001,
          "batch_size":32,
          "architecture":"AlexNet",
          "dataset":"custom"
          }

wandb.config = config
wandb.init(project="Traffic_sign_classifier",config=config)

path = '/kaggle/input/datasets/pavanvenky/traffic-signs/Traffic_Sign_Dataset/traffic_Data/DATA'
test_path = '/kaggle/input/datasets/pavanvenky/traffic-signs/Traffic_Sign_Dataset/traffic_Data/TEST'

fulldata,imagepaths,imagelabels = count_images_by_class(path)
testdata,imagepaths_test,imagelabels_test = count_images_by_class(test_path)

print(len(imagelabels))


class Mydata(Dataset):
    def __init__(self,images_data,images_labels,transform=None):
        self.imagesFolder = images_data
        self.imagesLabels = images_labels
        self.transform = transform

    def __len__(self):
        return len(self.imagesFolder)
    
    def __getitem__(self, index):
        img = cv2.imread(self.imagesFolder[index])
        if img is None:
            raise ValueError(f"Could not read image: {self.imagesFolder[index]}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        label = torch.tensor(self.imagesLabels[index], dtype=torch.long)

        if self.transform:
            img = self.transform(img)

        return img, label

transform = transforms.Compose([transforms.Resize((224,224)),
                                transforms.ToTensor()])

train_data = Mydata(imagepaths,imagelabels,transform)
train_data_loader = DataLoader(train_data,config["batch_size"],shuffle=True)

test_data = Mydata(imagepaths_test,imagelabels_test,transform)
test_data_loader = DataLoader(test_data,config["batch_size"],shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

# for images,lables in train_data_loader:
#     print(images.shape)
#     print(lables.shape)

# print("********")

# for images,lables in test_data_loader:
#     print(images.shape)
#     print(lables.shape)

num_classes = len(set(imagelabels))
model = Alexnet(num_classes=num_classes).to(device)

optim = torch.optim.AdamW(model.parameters(),config["lr"])
cost_fn = nn.CrossEntropyLoss()

# training start
epochs = config["epochs"]
total_step = len(train_data_loader)

wandb.watch(model,criterion=cost_fn,log="all",log_freq=100)

for epoch in range(epochs):
  model.train() 
  epoch_loss = 0.0
  for i, (images, labels) in enumerate(train_data_loader):
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)
      loss = cost_fn(outputs, labels)

      optim.zero_grad()
      loss.backward()
      optim.step()
      epoch_loss += loss.item()
      print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch+1, epochs, i+1, total_step, loss.item()))
      
  avg_epoch_loss = epoch_loss / total_step
  print(f"Epoch [{epoch+1}/{epochs}] Average Loss: {avg_epoch_loss:.4f}")

  if wandb is not None:
      wandb.log({"epoch": epoch + 1, "train_loss": avg_epoch_loss})


#Test pipeline
model.eval()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pavanvenkykaja (pavanvenkykaja-continental) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


class 0 (Speed limit (5km/h)): 118 images
class 1 (Speed limit (15km/h)): 40 images
class 2 (Speed limit (30km/h)): 80 images
class 3 (Speed limit (40km/h)): 260 images
class 4 (Speed limit (50km/h)): 98 images
class 5 (Speed limit (60km/h)): 194 images
class 6 (Speed limit (70km/h)): 78 images
class 7 (speed limit (80km/h)): 152 images
*****************************
class 0 (Speed limit (5km/h)): 14 images
class 1 (Speed limit (15km/h)): 12 images
class 2 (Speed limit (30km/h)): 60 images
class 3 (Speed limit (40km/h)): 84 images
class 4 (Speed limit (50km/h)): 58 images
class 5 (Speed limit (60km/h)): 50 images
class 6 (Speed limit (70km/h)): 30 images
class 7 (speed limit (80km/h)): 50 images
*****************************
1020
cuda
Epoch [1/250], Step [1/32], Loss: 2.1438
Epoch [1/250], Step [2/32], Loss: 19.3862
Epoch [1/250], Step [3/32], Loss: 9.5362
Epoch [1/250], Step [4/32], Loss: 9.6061
Epoch [1/250], Step [5/32], Loss: 8.9950
Epoch [1/250], Step [6/32], Loss: 5.3051
Epoch [1/

Alexnet(
  (layer1): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer3): Sequential(
    (0): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (layer4): Sequential(
    (0): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  

In [3]:
correct = 0.0
total = 0.0
for i,(images,lables) in enumerate(test_data_loader):
    images = images.to(device)
    labels = lables.to(device)
    test_output = model(images)
    probabilities = torch.softmax(test_output,dim=1)
    _, predicted = torch.max(probabilities, dim=1)
    
    correct += (predicted==labels).sum().item()
    total += lables.size(0)

accuracy = 100.0 * correct/total
print(f"Accuracy of the network on {total} test images: {accuracy:.2f}%")

Accuracy of the network on 358.0 test images: 81.56%
